# SecurePay AI — Real-Time Anomaly Detection in Financial Transactions
## Academic & Industry-Grade ML Research Monograph: Phase 1, Phase 2 & Phase 3 Execution

---

### Overview & Progress Status
- **Phase 0:** Architecture, API contracts, frontend/backend foundation [COMPLETE]
- **Phase 1:** Exploratory Data Analysis & Robust Preprocessing Pipeline [COMPLETE]
- **Phase 2:** Unsupervised Isolation Forest Anomaly Detection [COMPLETE]
- **Phase 3:** Local Outlier Factor (LOF) & Unsupervised Density Comparison [COMPLETE]
- **Phase 4:** Dynamic Threshold Calibration & Operating Points [SCHEDULED NEXT]

**Key Phase 3 Findings (Isolation Forest vs. Local Outlier Factor):**
- **Isolation Forest Superiority:** $F_1 = 0.2163$, $\text{PR-AUC} = 0.1554$, $\text{FP} = 572$, $\text{Latency} = 9.46\text{s}$.
- **LOF Underperformance:** $F_1 = 0.0691$, $\text{PR-AUC} = 0.0271$, $\text{FP} = 2,638$, $\text{Latency} = 108.23\text{s}$.
- **Curse of Dimensionality:** In 30-dimensional space, metric concentration degrades $k$-NN reachability distances relative to tree partitioning.

## 1. Environment Setup & Library Ingestion

In [1]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Visual style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10

print(f"Python Version: {sys.version.split()[0]}")
print(f"Pandas: {pd.__version__} | NumPy: {np.__version__}")

## 2. Dataset Loading & Path Verification
Ingestion of `data/creditcard.csv` in strict read-only mode.

In [2]:
DATA_PATH = Path("../data/creditcard.csv")
assert DATA_PATH.exists(), f"Target dataset not found at {DATA_PATH}"
df = pd.read_csv(DATA_PATH)
print(f"Dataset successfully loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Physical size: {DATA_PATH.stat().st_size:,} bytes")

## 3. Dataset Structural Overview
Inspection of dimensions, schema datatypes, and initial observations.

In [3]:
print("DataFrame Dimensions:", df.shape)
print("\nData Types Summary:")
print(df.dtypes.value_counts())
print(f"Memory usage: {df.memory_usage(deep=True).sum() / (1024*1024):.2f} MB")
df.head()

## 4. Missing Values Audit
Verification of dataset completeness across all 31 features.

In [4]:
null_counts = df.isnull().sum()
total_nulls = null_counts.sum()
print(f"Total missing values across all columns: {total_nulls}")
assert total_nulls == 0, "Integrity violation: null values detected!"
print("Data completeness: 100.00%")

## 5. Duplicate Row Analysis & Methodological Decision
Examination of 1,081 identical rows in the raw dataset.

In [5]:
total_duplicates = df.duplicated().sum()
class_dup_counts = df[df.duplicated()]['Class'].value_counts()
print(f"Total duplicate rows (excluding first occurrence): {total_duplicates:,} ({total_duplicates/len(df)*100:.4f}%)")
print(f"- Duplicates in Class 0 (Legitimate): {class_dup_counts.get(0, 0):,}")
print(f"- Duplicates in Class 1 (Fraudulent): {class_dup_counts.get(1, 0):,}")

# Verify whether any identical feature vectors carry contradictory labels
feature_cols = [c for c in df.columns if c != 'Class']
contradictions = df[df.duplicated(subset=feature_cols, keep=False)].groupby(feature_cols)['Class'].nunique()
print(f"Contradictory label cases: {(contradictions > 1).sum()}")

# Methodological decision note
print("\n[METHODOLOGICAL DECISION]: Raw CSV remains 100% untouched. Deduplication is documented and recommendable before train/test splitting to prevent identical cluster leakage.")

## 6. Extreme Class Imbalance & Evaluation Metric Validity
Quantification of severe class asymmetry ($1 : 578$) and mathematical invalidity of standard accuracy.

In [6]:
class_counts = df['Class'].value_counts()
legit = class_counts[0]
fraud = class_counts[1]
total = len(df)

print(f"Legitimate (Class 0): {legit:,} ({legit/total*100:.5f}%)")
print(f"Fraudulent (Class 1): {fraud:,} ({fraud/total*100:.5f}%)")
print(f"Imbalance Ratio: 1 : {legit/fraud:.1f}")

# Accuracy fallacy demonstration
trivial_accuracy = legit / total * 100
print(f"\nAccuracy Fallacy: A trivial zero-rule model classifying everything as normal achieves {trivial_accuracy:.3f}% accuracy but detects 0% of fraud.")
print("Required Evaluation Metrics: Precision, Recall, F1-Score, and Precision-Recall AUC.")

## 7. Temporal Analysis (Time Feature)
Analysis of 48-hour transaction velocity, diurnal oscillations, and off-peak fraud activity.

In [7]:
time_hours = df['Time'] / 3600.0
print(f"Time span: {time_hours.min():.2f} to {time_hours.max():.2f} hours (48 hours total)")
print(f"Median Time: {df['Time'].median():.2f}s | IQR: {df['Time'].quantile(0.75) - df['Time'].quantile(0.25):.2f}s")

# Diurnal pattern summary
legit_time = df[df['Class'] == 0]['Time'] / 3600.0
fraud_time = df[df['Class'] == 1]['Time'] / 3600.0
print(f"Legitimate peak/trough cycle: distinct sleep troughs at ~3-6h and ~27-30h.")
print(f"Fraud timing: uniform distribution with sustained activity during nighttime cardholder inactivity.")

## 8. Monetary Analysis (Amount Feature & Heavy Skewness)
Empirical investigation of right-skewed transaction values.

In [8]:
amount_stats = df['Amount'].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.99])
print("Transaction Amount Quantiles:")
print(amount_stats)
print(f"Amount Skewness: {df['Amount'].skew():.2f}")

print(f"\nLegitimate Median: ${df[df['Class']==0]['Amount'].median():.2f} (Mean: ${df[df['Class']==0]['Amount'].mean():.2f}, Max: ${df[df['Class']==0]['Amount'].max():.2f})")
print(f"Fraudulent Median: ${df[df['Class']==1]['Amount'].median():.2f} (Mean: ${df[df['Class']==1]['Amount'].mean():.2f}, Max: ${df[df['Class']==1]['Amount'].max():.2f})")

## 9. PCA Features Analysis (V1 through V28)
Inspection of orthogonal principal components resulting from confidentiality transformations.

In [9]:
pca_cols = [f"V{i}" for i in range(1, 29)]
pca_means = df[pca_cols].mean().abs().max()
pca_stds = df[pca_cols].std()
print(f"Max absolute mean across V1–V28: {pca_means:.4e} (zero-centered)")
print(f"Standard deviation range across V1–V28: [{pca_stds.min():.4f}, {pca_stds.max():.4f}]")
print("Note: No redundant PCA transformation applied; components are already orthogonal.")

## 10. Correlation Analysis & Feature Associations
Linear association audit between features and target variable `Class`.

In [10]:
corr = df.corr()
class_corr = corr['Class'].drop('Class').sort_values()
print("Top Negative Correlations with Fraud (Class):")
for feat, val in class_corr.head(5).items():
    print(f"  {feat}: {val:+.4f}")

print("\nTop Positive Correlations with Fraud (Class):")
for feat, val in class_corr.tail(5).items():
    print(f"  {feat}: {val:+.4f}")

## 11. Preprocessing Strategy & RobustScaler Justification
Mathematical comparison of standard vs robust scaling under extreme monetary skewness:

- **StandardScaler:** $z = \frac{x - \mu}{\sigma}$ is heavily distorted by $\$25,691.16$ outliers (inflating $\sigma=250.12$).
- **RobustScaler:** $x' = \frac{x - \text{median}}{\text{IQR}}$ relies on empirical medians ($\$22.00$) and interquartile ranges ($\$71.57$), preserving genuine outlier distance without suffering from parameter distortion.

## 12. Implementation of TransactionPreprocessor
Executing the reusable preprocessing module from `backend/ml/preprocessing/scaler.py`.

In [11]:
import sys
sys.path.append('..')
from backend.ml.preprocessing.scaler import TransactionPreprocessor

# Initialize preprocessor
preprocessor = TransactionPreprocessor(scale_amount=True, scale_time=True)

# Prevent data leakage: fit strictly on normal baseline transactions (Class == 0)
normal_df = df[df['Class'] == 0]
preprocessor.fit(normal_df)

print(f"Preprocessor successfully fitted: {preprocessor.is_fitted}")
print(f"Fitted Amount Median: ${preprocessor.amount_median_:.2f} | IQR: ${preprocessor.amount_iqr_:.2f}")
print(f"Fitted Time Median: {preprocessor.time_median_:.2f}s | IQR: {preprocessor.time_iqr_:.2f}s")

# Demonstrate transformation on mixed test sample
sample_records = df.head(5)
transformed_sample = preprocessor.transform(sample_records)
print(f"Input shape: {sample_records.shape} -> Transformed shape: {transformed_sample.shape}")
print(f"Target 'Class' excluded from model features: {transformed_sample.shape[1] == 30}")

## 13. Training Data Partitioning Strategy
Decoupling the normal behavior baseline for unsupervised training from the mixed evaluation set.

In [12]:
# Train partition: Unsupervised models learn strictly from legitimate purchases (Class == 0)
X_train_normal = df[df['Class'] == 0].drop(columns=['Class'])
y_train_normal = df[df['Class'] == 0]['Class']

# Evaluation partition: Mixed streaming dataset containing both legitimate and fraudulent records
X_eval_mixed = df.drop(columns=['Class'])
y_eval_mixed = df['Class']

print(f"Unsupervised Normal Training Pool: {len(X_train_normal):,} samples")
print(f"Combined Evaluation Pool: {len(X_eval_mixed):,} samples ({y_eval_mixed.sum()} fraud events)")
print("[PHASE 1 COMPLETE]: Preprocessing pipeline and EDA validated. Models scheduled for Phase 2.")

---
# PHASE 2: UNSUPERVISED ISOLATION FOREST ANOMALY DETECTION

## 14. Unsupervised Anomaly Detection Framing & Motivation

In enterprise payment networks, financial fraud detection encounters three fundamental challenges that limit purely supervised classification:

1. **Extreme Class Asymmetry:** Legitimate transactions comprise $99.827\%$ of traffic, while fraud represents only $0.173\%$ ($1 : 578$). Supervised models risk extreme bias toward majority class consensus.
2. **Adversarial Non-Stationarity (Concept Drift):** Fraud syndicates continuously alter behavior, evasion techniques, and transaction amounts. Supervised algorithms trained exclusively on historical fraud patterns fail to generalize against novel ('zero-day') attack vectors.
3. **Verification Latency & Delayed Ground Truth:** Chargebacks, consumer fraud disputes, and forensic investigation take weeks or months to confirm labels. An unsupervised model trained on verified normal baselines operates without requiring real-time target labels.

**Core Architectural Hypothesis:**
By profiling legitimate cardholder behavior exclusively, fraudulent transactions manifest as topological outliers residing in sparse, low-density regions of the 30-dimensional feature space.

## 15. Isolation Forest Theory & Mathematical Derivations

Isolation Forest (Liu, Ting, and Zhou, 2008) operates on the principle that anomalies are *few and different*. Instead of estimating dense clusters or distance matrices, Isolation Forest explicitly isolates instances using recursive random partitioning.

### 15.1 Recursive Space Partitioning ($iTree$)
Given a dataset $X = \{x_1, \dots, x_n\}$ with $d$ dimensions:
1. A feature $q \in \{1, \dots, d\}$ is randomly selected.
2. A split value $p$ is uniformly chosen at random between $\min(X_{\cdot, q})$ and $\max(X_{\cdot, q})$.
3. The sample is recursively partitioned until either:
   - The tree reaches maximum depth limit: $h_{\max} = \lceil \log_2(n) \rceil$.
   - The node contains $|X| \le 1$.
   - All data points at the node have identical attribute values.

Because anomalies occupy sparse regions, they require significantly fewer partitions to be isolated compared to normal instances clustered in high-density regions.

### 15.2 Path Length & Average Search Cost ($c(n)$)
The path length $h(x)$ of an instance $x$ is the number of edges traversed from the root node to the terminating leaf node in an isolation tree ($iTree$).

Because $iTree$ structures share equivalent properties with Binary Search Trees (BST), the average path length of an unsuccessful search in a BST constructed over $n$ items serves as the normalization constant $c(n)$:

$$c(n) = 2 \left( \ln(n - 1) + \gamma \right) - \frac{2(n - 1)}{n}$$

where $\gamma \approx 0.5772156649$ is the Euler-Mascheroni constant.

### 15.3 Continuous Anomaly Score Formulation ($s(x, n)$)
The anomaly score $s(x, n)$ across an ensemble of $k$ isolation trees is defined as:

$$s(x, n) = 2^{-\frac{\mathbb{E}(h(x))}{c(n)}}$$

where $\mathbb{E}(h(x)) = \frac{1}{k} \sum_{i=1}^k h_i(x)$ is the expected path length across all $k$ trees.

- If $\mathbb{E}(h(x)) \to 0 \implies s(x, n) \to 1$: The point is rapidly isolated near the root $\implies$ **Strong Anomaly**.
- If $\mathbb{E}(h(x)) \to n - 1 \implies s(x, n) \to 0$: The point requires maximum recursive splits $\implies$ **Normal Instance**.
- If $\mathbb{E}(h(x)) \to c(n) \implies s(x, n) \to 0.5$: Indistinguishable from the general sample mean.

### 15.4 Standardized Scoring Convention
To provide a consistent, interpretable metric across the SecurePay AI platform:

$$\text{anomaly\_score}(x) = -\text{score\_samples}(x)$$

Higher continuous scores indicate higher anomaly likelihood, bounded and calibrated for real-time alerting.

## 16. Legitimate-Only Training Partition & Preprocessing Execution

We construct the clean normal training baseline:
1. Filter strictly for legitimate transactions: `Class == 0` (284,315 rows).
2. Remove identical duplicates (1,062 rows) $\implies$ **283,253 training samples**.
3. Decouple `Class` completely to avoid target leakage.
4. Fit `TransactionPreprocessor` (`RobustScaler` on `Amount` and `Time`).

In [13]:
import time
from backend.ml.preprocessing.scaler import TransactionPreprocessor
from backend.ml.isolation_forest.model import IsolationForestDetector
from backend.ml.evaluation.metrics import AnomalyEvaluator

# 1. Extract legitimate transactions
df_legit = df[df['Class'] == 0].copy()
print(f"Raw legitimate instances: {len(df_legit):,}")

# 2. Apply deduplication to training pool
df_train_dedup = df_legit.drop_duplicates()
print(f"Deduplicated legitimate training pool: {len(df_train_dedup):,} (removed {len(df_legit) - len(df_train_dedup):,} duplicates)")

# 3. Fit transaction preprocessor strictly on normal training data
preprocessor = TransactionPreprocessor(scale_amount=True, scale_time=True)
X_train_df = df_train_dedup.drop(columns=['Class'])
preprocessor.fit(X_train_df)
X_train_scaled = preprocessor.transform(X_train_df)

# 4. Prepare complete evaluation dataset (all 284,807 transactions with ground-truth Class)
X_eval_df = df.drop(columns=['Class'])
y_eval = df['Class'].to_numpy(dtype=int)
X_eval_scaled = preprocessor.transform(X_eval_df)

print(f"Training matrix shape: {X_train_scaled.shape}")
print(f"Evaluation matrix shape: {X_eval_scaled.shape}")
print(f"Evaluation fraud instances: {y_eval.sum():,} ({y_eval.mean()*100:.3f}%)")

## 17. Contamination Hyperparameter Experimentation Grid

The `contamination` parameter defines the proportion of anomalies the model expects during decision boundary thresholding.
We systematically evaluate seven candidate values across orders of magnitude:
$$\mathcal{C} = \{0.001, 0.002, 0.005, 0.010, 0.020, 0.030, 0.050\}$$

Each candidate is trained with $n\_estimators = 200$, $max\_samples = 256$, and $random\_state = 42$.

In [14]:
contamination_grid = [0.001, 0.002, 0.005, 0.01, 0.02, 0.03, 0.05]
experiment_results = []

for c in contamination_grid:
    t0 = time.perf_counter()
    model = IsolationForestDetector(
        n_estimators=200,
        contamination=c,
        max_samples=256,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train_scaled)
    fit_duration = time.perf_counter() - t0
    
    # Predict on complete evaluation set
    y_pred = model.predict(X_eval_scaled)
    anomaly_scores = model.score_samples(X_eval_scaled)
    
    # Compute quantitative anomaly metrics
    metrics = AnomalyEvaluator.evaluate_binary(y_eval, y_pred, scores=anomaly_scores)
    metrics['contamination'] = c
    metrics['execution_time_sec'] = round(fit_duration, 2)
    experiment_results.append(metrics)
    
    print(f"c={c:.3f} | Anom: {metrics['predicted_anomalies']:5d} | TP: {metrics['tp']:3d} | FP: {metrics['fp']:5d} | "
          f"P: {metrics['precision']:.4f} | R: {metrics['recall']:.4f} | F1: {metrics['f1']:.4f} | Time: {fit_duration:.2f}s")

df_experiments = pd.DataFrame(experiment_results)
cols_order = ['contamination', 'predicted_anomalies', 'tp', 'fp', 'fn', 'tn', 'precision', 'recall', 'f1', 'pr_auc', 'execution_time_sec']
df_experiments = df_experiments[cols_order]

## 18. Quantitative Evaluation & Empirical Benchmark Table

The empirical results across the contamination grid reveal the fundamental precision-recall trade-off curve:

In [15]:
print("=== ISOLATION FOREST CONTAMINATION EXPERIMENTS BENCHMARK ===")
display(df_experiments)

best_f1_idx = df_experiments['f1'].idxmax()
best_config = df_experiments.loc[best_f1_idx]
print(f"\nOptimal F1 Configuration: Contamination = {best_config['contamination']}")
print(f"Precision: {best_config['precision']:.4f} | Recall: {best_config['recall']:.4f} | F1-Score: {best_config['f1']:.4f}")
print(f"Confusion Matrix: TP={int(best_config['tp'])}, FP={int(best_config['fp'])}, FN={int(best_config['fn'])}, TN={int(best_config['tn'])}")

## 19. Confusion Matrix Visualization & Empirical Error Analysis

We inspect the confusion matrix of the optimal baseline model ($c = 0.002$).

In [16]:
from sklearn.metrics import confusion_matrix

# Train optimal production candidate model
optimal_model = IsolationForestDetector(
    n_estimators=200,
    contamination=0.002,
    max_samples=256,
    random_state=42,
    n_jobs=-1
)
optimal_model.fit(X_train_scaled)
y_pred_opt = optimal_model.predict(X_eval_scaled)
scores_opt = optimal_model.score_samples(X_eval_scaled)

cm = confusion_matrix(y_eval, y_pred_opt)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(6, 5), dpi=150)
sns.heatmap(
    cm,
    annot=True,
    fmt=",d",
    cmap="Blues",
    cbar=False,
    xticklabels=["Pred Normal (0)", "Pred Anomaly (1)"],
    yticklabels=["Actual Normal (0)", "Actual Fraud (1)"],
    annot_kws={"fontsize": 12, "weight": "bold"}
)
ax.set_title("Isolation Forest Confusion Matrix (Contamination = 0.002)", fontsize=12, pad=12)
plt.tight_layout()
plt.show()

## 20. Continuous Anomaly Score Distribution Analysis & PR-AUC

We analyze the empirical distribution of continuous anomaly scores $\text{anomaly\_score} = -\text{score\_samples}(X)$ between legitimate and fraudulent transactions.

In [17]:
from sklearn.metrics import precision_recall_curve, auc

scores_legit = scores_opt[y_eval == 0]
scores_fraud = scores_opt[y_eval == 1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

# Subplot 1: Score Distributions
sns.kdeplot(scores_legit, ax=ax1, label='Legitimate (Class 0)', color='#1e3a8a', fill=True, alpha=0.3)
sns.kdeplot(scores_fraud, ax=ax1, label='Fraudulent (Class 1)', color='#dc2626', fill=True, alpha=0.5)
ax1.axvline(optimal_model.threshold_, color='black', linestyle='--', label=f'Threshold ({optimal_model.threshold_:.4f})')
ax1.set_xlabel('Anomaly Score (-score_samples)')
ax1.set_ylabel('Density')
ax1.set_title('Continuous Anomaly Score Distribution by Class')
ax1.legend()

# Subplot 2: Precision-Recall Curve
precision_curve, recall_curve, _ = precision_recall_curve(y_eval, scores_opt)
pr_auc_val = auc(recall_curve, precision_curve)
ax2.plot(recall_curve, precision_curve, color='#2563eb', lw=2, label=f'PR-Curve (AUC = {pr_auc_val:.4f})')
ax2.axhline(y_eval.mean(), color='gray', linestyle=':', label=f'No-Skill Baseline ({y_eval.mean():.4f})')
ax2.set_xlabel('Recall (Fraud Detection Rate)')
ax2.set_ylabel('Precision (Positive Predictive Value)')
ax2.set_title('Precision-Recall Curve (Continuous Anomaly Scores)')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"Legitimate Score Median: {np.median(scores_legit):.4f} (IQR: {np.percentile(scores_legit, 75) - np.percentile(scores_legit, 25):.4f})")
print(f"Fraudulent Score Median: {np.median(scores_fraud):.4f} (IQR: {np.percentile(scores_fraud, 75) - np.percentile(scores_fraud, 25):.4f})")
print(f"Separation Distance: {np.median(scores_fraud) - np.median(scores_legit):.4f}")
print(f"Precision-Recall AUC: {pr_auc_val:.4f}")

## 21. Business Trade-off Analysis: Precision vs. Recall

In institutional payments processing, the operational cost function is asymmetric:

$$\mathcal{C}_{\text{total}} = \text{Cost}_{\text{FN}} \cdot \text{FN} + \text{Cost}_{\text{FP}} \cdot \text{FP}$$

- **Cost of False Negatives (FN):** Unintercepted fraud leads to financial loss, dispute recovery fees, card reissuance, and chargeback penalties.
- **Cost of False Positives (FP):** Legitimate customer transactions declined create consumer friction, brand dissatisfaction, and call center load.

### Operating Point Trade-offs:
1. **Conservative Operating Point ($c = 0.002$):**
   - **FP = 572**, **TP = 129**, **Recall = $26.2\%$**, **Precision = $18.4\%$**.
   - Only $0.20\%$ of legitimate transactions are interrupted. Recommended for automated hard declines.
2. **Aggressive Audit Operating Point ($c = 0.050$):**
   - **FP = 14,355**, **TP = 418**, **Recall = $84.96\%$**, **Precision = $2.83\%$**.
   - Captures $85\%$ of all fraud, but flags $5.05\%$ of legitimate accounts. Recommended for secondary biometric step-up authentication (SMS OTP / 3D-Secure) rather than outright transaction rejection.

## 22. Model Serialization & Phase 2 Baseline Sign-off

We persist the trained model and preprocessor artifacts to `outputs/models/` for ingestion by the FastAPI service.

In [18]:
from pathlib import Path

MODELS_DIR = Path("../outputs/models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

preprocessor_path = MODELS_DIR / "preprocessor.joblib"
model_path = MODELS_DIR / "isolation_forest.joblib"

preprocessor.save(preprocessor_path)
optimal_model.save(model_path)

print(f"Saved preprocessor to: {preprocessor_path.resolve()}")
print(f"Saved Isolation Forest model to: {model_path.resolve()}")
print("\n[PHASE 2 COMPLETE]: Isolation Forest baseline established and persisted. Ready for Phase 3 (Local Outlier Factor).")

---
# PHASE 3: LOCAL OUTLIER FACTOR (LOF) ANOMALY DETECTION

## 23. Local Outlier Factor (LOF) Theory & Mathematical Formulations

Local Outlier Factor (Breunig et al., 2000) is a density-based unsupervised anomaly detection algorithm. Unlike global density approaches, LOF evaluates whether a data point has a significantly lower density than its immediate neighbors.

### 23.1 Mathematical Definitions
1. **$k$-distance of $p$:** The Euclidean distance $d(p, o)$ between observation $p$ and its $k$-th nearest neighbor $o \in X$.
2. **$k$-distance Neighborhood ($N_k(p)$):** The set of all objects whose distance from $p$ is at most $k$-distance($p$).
3. **Reachability Distance:**
   $$\text{reach\_dist}_k(p, o) = \max\left( k\text{-distance}(o), d(p, o) \right)$$
   This asymmetrical distance dampens fluctuations within dense clusters.
4. **Local Reachability Density (LRD):**
   $$\text{lrd}_k(p) = \frac{|N_k(p)|}{\sum_{o \in N_k(p)} \text{reach\_dist}_k(p, o)}$$
   The reciprocal of the average reachability distance from $p$ to its $k$-neighbors.
5. **Local Outlier Factor (LOF):**
   $$\text{LOF}_k(p) = \frac{\sum_{o \in N_k(p)} \frac{\text{lrd}_k(o)}{\text{lrd}_k(p)}}{|N_k(p)|}$$
   - $\text{LOF} \approx 1$: Point has comparable density to neighbors (Inlier / Legitimate).
   - $\text{LOF} < 1$: Point resides in a denser cluster interior.
   - $\text{LOF} > 1$: Point has substantially lower local density than neighbors $\implies$ **Outlier / Anomaly**.

### 23.2 Novelty Detection Formulation (`novelty=True`)
Standard LOF (`novelty=False`) operates strictly as an in-sample outlier detector with no out-of-sample inference. For streaming financial transactions, we configure `novelty=True`, fitting the $k$-NN index exclusively on legitimate training baseline transactions (`Class == 0`) and scoring unseen evaluation transactions.

### 23.3 Standardized Continuous Scoring Convention
In scikit-learn, `score_samples(X)` returns the *opposite LOF* (negative values where lower indicates higher abnormality). To ensure project-wide consistency where **higher score = more anomalous**, we define:
$$\text{anomaly\_score}(x) = -\text{score\_samples}(x)$$

## 24. Training Partition & Leakage Prevention Protocol

Consistent with Phase 2:
1. **Training Pool:** Legitimate transactions only (`Class == 0`, 283,253 samples deduplicated).
2. **Zero Leakage:** Ground-truth `Class` is strictly excluded from input vectors.
3. **Preprocessing:** RobustScaler applied to `Amount` and `Time`, preserving orthogonal PCA coordinates `V1`–`V28`.

In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from backend.ml.lof.model import LocalOutlierFactorDetector
from backend.ml.evaluation.metrics import AnomalyEvaluator

# Verify training shapes and leakage protocol
print(f"Legitimate baseline training samples: {len(X_train_scaled):,}")
print(f"Evaluation samples: {len(X_eval_scaled):,} (Fraud count: {y_eval.sum():,})")
print(f"Input feature dimensionality: {X_train_scaled.shape[1]} (Class column decoupled)")

## 25. Controlled Hyperparameter Experimentation Grid

We systematically evaluate $4 \times 4 = 16$ configurations across neighborhood sizes and contamination levels:
- $n\_neighbors \in [10, 20, 35, 50]$
- $contamination \in [0.001, 0.002, 0.005, 0.010]$

In [20]:
# Load empirical experiment results from outputs/metrics/lof_experiments.csv
lof_experiments_path = Path("../outputs/metrics/lof_experiments.csv")
df_lof_exp = pd.read_csv(lof_experiments_path)

print("=== LOCAL OUTLIER FACTOR CONTROLLED EXPERIMENT GRID (16 CONFIGURATIONS) ===")
display(df_lof_exp)

## 26. Optimal LOF Model Selection & Confusion Matrix

The top configuration by $F_1$-score and PR-AUC is:
- $n\_neighbors = 50$
- $contamination = 0.010$

Performance: $\text{Recall} = 22.76\%$, $\text{Precision} = 4.07\%$, $F_1 = 0.0691$, $\text{PR-AUC} = 0.0271$.

In [21]:
from sklearn.metrics import confusion_matrix

# Load serialized LOF model
lof_model = LocalOutlierFactorDetector.load("../outputs/models/local_outlier_factor.joblib")
print(f"Loaded LOF detector: k={lof_model.n_neighbors}, c={lof_model.contamination}")
print(f"Offset: {lof_model.offset_:.4f} | Standardized Threshold: {lof_model.threshold_:.4f}")

y_pred_lof = lof_model.predict(X_eval_scaled)
scores_lof = lof_model.score_samples(X_eval_scaled)

cm_lof = confusion_matrix(y_eval, y_pred_lof)
tn_l, fp_l, fn_l, tp_l = cm_lof.ravel()

fig, ax = plt.subplots(figsize=(6, 5), dpi=150)
sns.heatmap(
    cm_lof,
    annot=True,
    fmt=",d",
    cmap="Blues",
    cbar=False,
    xticklabels=["Pred Normal (0)", "Pred Anomaly (1)"],
    yticklabels=["Actual Normal (0)", "Actual Fraud (1)"],
    annot_kws={"fontsize": 12, "weight": "bold"}
)
ax.set_title(f"LOF Confusion Matrix (k={lof_model.n_neighbors}, c={lof_model.contamination})", fontsize=12, pad=12)
plt.tight_layout()
plt.show()

print(f"TP: {tp_l} | FP: {fp_l:,} | FN: {fn_l} | TN: {tn_l:,}")
print(f"Precision: {tp_l/(tp_l+fp_l):.4f} | Recall: {tp_l/(tp_l+fn_l):.4f}")

## 27. Precision-Recall Curve & Anomaly Score Distribution

Visualizing the Precision-Recall curve and score distribution between legitimate and fraudulent populations.

In [22]:
from sklearn.metrics import precision_recall_curve, auc

p_curve, r_curve, _ = precision_recall_curve(y_eval, scores_lof)
pr_auc_lof = auc(r_curve, p_curve)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

# Subplot 1: Score Distributions
scores_legit_l = scores_lof[y_eval == 0]
scores_fraud_l = scores_lof[y_eval == 1]
sns.kdeplot(scores_legit_l, ax=ax1, label='Legitimate (Class 0)', color='#141C2B', fill=True, alpha=0.3, cut=0)
sns.kdeplot(scores_fraud_l, ax=ax1, label='Fraudulent (Class 1)', color='#DC2626', fill=True, alpha=0.5, cut=0)
ax1.axvline(lof_model.threshold_, color='#2C4A8F', linestyle='--', label=f'Threshold ({lof_model.threshold_:.4f})')
ax1.set_xlabel('Standardized LOF Score (-score_samples)')
ax1.set_ylabel('Density')
ax1.set_title('LOF Anomaly Score Distribution by Class')
ax1.legend()

# Subplot 2: Precision-Recall Curve
ax2.plot(r_curve, p_curve, color='#2C4A8F', lw=2, label=f'LOF PR-Curve (AUC = {pr_auc_lof:.4f})')
ax2.axhline(y_eval.mean(), color='gray', linestyle=':', label=f'No-Skill Baseline ({y_eval.mean():.4f})')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Local Outlier Factor Precision-Recall Curve')
ax2.legend()

plt.tight_layout()
plt.show()

## 28. Algorithmic Head-to-Head Comparison: Isolation Forest vs. Local Outlier Factor

We compare both unsupervised models across quantitative discrimination metrics, false positive burdens, and computational cost.

In [23]:
comparison_data = {
    "Dimension": ["Algorithm Type", "Optimal Config", "Precision", "Recall", "F1-Score", "PR-AUC", "False Positives", "False Negatives", "Execution Latency", "Model Size"],
    "Isolation Forest": ["Recursive Tree Partitioning", "c=0.002, n_trees=200", "18.40%", "26.22%", "0.2163", "0.1554", "572", "363", "~9.46s", "1.84 MB"],
    "Local Outlier Factor": ["Density-based k-NN", "k=50, c=0.010", "4.07%", "22.76%", "0.0691", "0.0271", "2,638", "380", "~108.23s", "185.8 MB"]
}
df_comp = pd.DataFrame(comparison_data)
display(df_comp)

## 29. Analytical Diagnosis: Why Does Isolation Forest Outperform LOF in Financial Fraud?

Our empirical experiments establish that **Isolation Forest outperforms Local Outlier Factor by $3.1\times$ in F1-score ($0.2163$ vs $0.0691$) and $5.7\times$ in PR-AUC ($0.1554$ vs $0.0271$)** while generating **$4.6\times$ fewer False Positives (572 vs 2,638)**. Three fundamental mathematical factors explain this divergence:

1. **The Curse of Dimensionality & Metric Concentration:**
   In 30-dimensional continuous space, Euclidean distance suffers from metric concentration: the distance between the nearest neighbor and farthest neighbor converges towards a narrow relative band. As a result, reachability distances become less distinct, eroding local density contrasts.
2. **Multi-Modal Density Variations Across Financial Regimes:**
   Financial transaction volumes span multiple orders of magnitude across time of day and purchase sizes. Legitimate transactions naturally exhibit variable densities in different sub-spaces. LOF struggles to establish a unified reachability threshold across heterogeneous clusters.
3. **Computational Scalability & Memory Footprint:**
   Isolation Forest evaluates sub-samples ($256$ points) per tree in $O(n \log n)$ time with compact tree storage ($1.84$ MB). LOF requires retaining the entire $283,253$-point reference index ($185.8$ MB) and computing nearest neighbor queries across 30 dimensions for every single transaction, incurring severe inference latency unsuitable for millisecond fraud gateways.

## 30. Phase 3 Baseline Persistence & Phase 4 Transition

- Both unsupervised models are persisted to `outputs/models/`.
- Complete experiment logs recorded in `outputs/metrics/`.
- FastAPI backend service updated to serve real-time predictions for both architectures.
- **Next Phase (Phase 4):** Dynamic threshold optimization, cost-sensitive operating points, and interactive web dashboard integration.

---
# METHODOLOGICAL EVALUATION AUDIT & HOLDOUT GENERALIZATION VALIDATION

## 31. Evaluation Protocol Audit & Methodological Clarification

Before embarking on Phase 4 (Threshold Calibration & Operating Points), we conducted a rigorous **methodological audit** of the evaluation protocols employed across Phase 2 and Phase 3.

### 31.1 Assessment of Historical Baseline Evaluation Protocol
In Phase 2 and Phase 3:
- Models were trained on the legitimate baseline partition (`Class == 0`, 283,253 samples deduplicated).
- Models were then evaluated across the complete dataset (`df`, 284,807 transactions).

**Audit Finding:**
Describing the complete-dataset evaluation as strictly *"out-of-sample"* was **technically inaccurate for the legitimate partition**. Because the 283,253 legitimate training instances were included in the evaluation set, the evaluation was *in-sample* for those legitimate observations, and truly *out-of-sample* only for the 492 fraud events and the 1,062 legitimate duplicates. In enterprise and academic settings, this protocol is properly classified as a **Full Population / Semi-Supervised Baseline**.

### 31.2 New Rigorous Holdout Evaluation Protocol (60% / 20% / 20% Stratified Split)
To measure true, unbiased generalization to unseen future transactions without evaluation overlap:
1. **Stratified Split (`random_state = 42`):**
   - **Train Set (60%):** 170,884 transactions (170,589 legitimate, 295 fraud).
   - **Validation Set (20%):** 56,961 transactions (56,863 legitimate, 98 fraud).
   - **Test Set (20%):** 56,962 transactions (56,863 legitimate, 99 fraud).
2. **Strict Unsupervised Training Restriction:**
   - Unsupervised models are fitted **exclusively on legitimate transactions from the training partition** (`Class == 0`, deduplicated: **170,188 samples**).
   - All 295 fraud cases in the training split are excluded from model training to prevent target leakage.
3. **Zero Preprocessing Leakage:**
   - `TransactionPreprocessor` is fitted **exclusively on the 170,188 training legitimate samples**. Neither validation nor test records are ever seen by the scaler.
4. **Untouched Test Evaluation:**
   - The 20% test partition (56,962 transactions; 99 fraud cases) is held out and evaluated **exactly once**.

In [24]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Load audit split summary
val_dir = Path("../outputs/validation")
with open(val_dir / "evaluation_split_summary.json", "r") as f:
    audit_split = json.load(f)

print("=== STRATIFIED HOLDOUT PARTITION METRICS ===")
print(f"Total Observations: {audit_split['split_dimensions']['total_dataset_rows']:,}")
print(f"Train Partition:      {audit_split['split_dimensions']['train_partition']['total_rows']:,} rows (Unsupervised pool: {audit_split['split_dimensions']['train_partition']['unsupervised_train_pool_dedup']:,} clean normal samples)")
print(f"Validation Partition: {audit_split['split_dimensions']['validation_partition']['total_rows']:,} rows ({audit_split['split_dimensions']['validation_partition']['fraud_rows']} fraud cases)")
print(f"Test Partition:       {audit_split['split_dimensions']['test_partition']['total_rows']:,} rows ({audit_split['split_dimensions']['test_partition']['fraud_rows']} fraud cases)")
print(f"Test Fraud Prevalence (No-Skill Baseline): {audit_split['split_dimensions']['test_partition']['fraud_prevalence']*100:.4f}%")

## 32. Holdout Test Set Evaluation & Direct Algorithmic Comparison

Both models were trained on the exact same 170,188 legitimate training samples and evaluated on the exact same untouched holdout test partition (56,962 samples; 99 fraud cases).

In [25]:
# Load holdout comparison CSV
df_test_comp = pd.read_csv(val_dir / "model_comparison_test.csv")
print("=== UNTOUCHED HOLDOUT TEST SET PERFORMANCE (N = 56,962, FRAUD = 99) ===")
display(df_test_comp[["model", "config", "tp", "fp", "fn", "precision", "recall", "f1_score", "pr_auc", "fpr"]])

## 33. Comparative Diagnosis on Holdout Test Partition

The holdout validation provides essential insights:

1. **Isolation Forest Generalization:**
   - **Test Metrics:** Precision = $17.39\%$, Recall = $24.24\%$, $F_1 = 0.2025$, $\text{PR-AUC} = 0.1081$, $\text{FPR} = 0.20\%$ (114 False Positives).
   - Performance closely mirrors its full-population baseline ($F_1 = 0.2163$, Recall = $26.22\%$, Precision = $18.40\%$), demonstrating remarkable architectural stability and minimal variance across sample sizes.
2. **Local Outlier Factor Sample-Size Sensitivity:**
   - **Test Metrics:** Precision = $12.88\%$, Recall = $81.82\%$, $F_1 = 0.2225$, $\text{PR-AUC} = 0.2186$, $\text{FPR} = 0.96\%$ (548 False Positives).
   - On the 170k training sample, LOF captures $81$ out of $99$ holdout frauds ($81.82\%$ recall), but flags $548$ legitimate transactions ($4.8\times$ more false alarms than Isolation Forest).
3. **PR-AUC Calculation Fidelity:**
   - Both PR-AUC values are computed from continuous anomaly scores ($	ext{score} = -\text{score\_samples}(X)$), where higher scores denote greater anomaly likelihood.
   - Both models perform dramatically above the no-skill baseline ($0.1738\%$): Isolation Forest is $62\times$ higher, and LOF is $126\times$ higher.

# PHASE 4 — DYNAMIC THRESHOLD CALIBRATION, OPERATING POINTS & FINAL MODEL RECOMMENDATION

### Methodological Hierarchy & Terminology Demarcation
To preserve complete scientific integrity, this project strictly distinguishes between four evaluation stages:
1. **Historical Baseline (Phases 2 & 3):** Full-population (N=284,807) unsupervised exploratory experiments establishing reference behavior.
2. **Holdout Validation Audit:** Leakage-safe 60/20/20 stratified split (`random_state=42`), verifying scaling without data snooping.
3. **Phase 4 Threshold Calibration:** Continuous score cutoff selection conducted **strictly on the Validation partition (N=56,961)** across 216 quantile points.
4. **Final Test Evaluation:** Fixed operating points evaluated **exactly once on the untouched Test partition (N=56,962; 99 fraud cases)** with zero test-based tuning.

## 34. Why Dynamic Threshold Calibration is Necessary

Default decision cutoffs in unsupervised anomaly detection (e.g. `offset_ = -0.5` or `contamination = 0.002`) are arbitrary mathematical priors, not operationally tuned boundaries.

In commercial financial fraud detection:
* An overly conservative cutoff misses fraudulent attacks (False Negatives), incurring direct chargeback liability and regulatory penalties.
* An overly aggressive cutoff flags thousands of normal transactions (False Positives), causing high operational investigation expense and severe cardholder insult/friction.
* **Dynamic calibration** allows compliance and risk officers to select deliberate operating points tailored to specific operational contexts (e.g., automated blocking vs. SMS 2FA step-up vs. manual batch review).

In [ ]:
# Load calibrated operating points and sweep summaries
import json
import pandas as pd
import matplotlib.pyplot as plt

with open('outputs/thresholds/selected_operating_points.json', 'r') as f:
    operating_points = json.load(f)

with open('outputs/thresholds/business_cost_analysis.json', 'r') as f:
    cost_analysis = json.load(f)

if_sweep_df = pd.read_csv('outputs/thresholds/isolation_forest_thresholds.csv')
lof_sweep_df = pd.read_csv('outputs/thresholds/lof_thresholds.csv')

print(f"Loaded {len(if_sweep_df)} validation thresholds for Isolation Forest")
print(f"Loaded {len(lof_sweep_df)} validation thresholds for Local Outlier Factor")
print(f"Test Fraud Prevalence: {operating_points['metadata']['test_fraud_prevalence']:.6f} ({operating_points['metadata']['test_fraud_prevalence']*100:.4f}%)")

## 35. Continuous Anomaly Scores Convention

Both models use standard continuous anomaly scores where **higher score = more anomalous**:
* **Isolation Forest:** $s(x) = -\text{score\_samples}(x) \in [0, 1]$. Shorter average tree path length yields a higher anomaly score.
* **Local Outlier Factor:** $s(x) = -\text{negative\_outlier\_factor\_} \in [1, \infty)$. Lower local reachability density relative to neighbors yields a higher score ($s(x) > 1$ represents an outlier manifold).

Continuous scores allow dense quantile sweeps across the entire empirical support distribution, completely avoiding the limitations of binary classification outputs.

## 36. Validation Dense Threshold Sweep & PR Curves

A dense sweep of 216 quantile points was conducted on the **Validation partition (N=56,961)** to map the full Precision-Recall and ROC geometry.

Key Validation Findings:
* **Isolation Forest Validation PR-AUC:** 0.1260 ($72.5\times$ enrichment over no-skill baseline of 0.001738).
* **Local Outlier Factor Validation PR-AUC:** 0.2319 ($133.4\times$ enrichment over no-skill baseline).
* Across the entire curve, LOF delivers systematically higher recall for any chosen precision constraint.

In [ ]:
# Display Validation Precision-Recall Curves figure
from IPython.display import Image, display
display(Image(filename='outputs/figures/validation_pr_curves.png', width=750))
display(Image(filename='outputs/figures/validation_precision_recall.png', width=750))

## 37. False Positive Rate (FPR) vs Recall Trade-offs

In high-volume payment processing (e.g., millions of daily card transactions), even small False Positive Rates generate severe operational load:
* $\text{FPR} = 1.0\%$ on $10^6$ daily transactions represents **10,000 false alarms/day**.
* $\text{FPR} \le 0.25\%$ represents **\le 2,500 false alarms/day**.

Our validation sweep establishes the empirical Pareto frontier between FPR and Recall for both models:

In [ ]:
# Display FPR vs Recall and F1 vs Threshold figures
display(Image(filename='outputs/figures/validation_fpr_recall.png', width=750))
display(Image(filename='outputs/figures/validation_f1_threshold.png', width=750))

## 38. Configurable Business Cost Analysis

> **Methodological Disclaimer:** Cost parameters are illustrative modeling assumptions for operational sensitivity analysis, not proprietary SecurePay financial figures.

The objective function is parameterized as:
$$\text{Total Cost} = \text{FP} \times C_{\text{FP}} + \text{FN} \times C_{\text{FN}}$$

Three canonical risk scenarios were evaluated on the validation sweep:
1. **Balanced Operational Cost (1:10):** $C_{\text{FP}}=1, C_{\text{FN}}=10$.
2. **High Loss Asymmetry (1:20):** $C_{\text{FP}}=1, C_{\text{FN}}=20$.
3. **Severe Fraud Penalty (1:50):** $C_{\text{FP}}=1, C_{\text{FN}}=50$.

In [ ]:
# Display Business Cost vs Threshold figure
display(Image(filename='outputs/figures/validation_business_cost.png', width=750))

# Summary table of cost scenarios
cost_rows = []
for sc in cost_analysis['scenarios']:
    cost_rows.append({
        'Scenario': sc['scenario'],
        'Cost Ratio (FP:FN)': f"{sc['cost_fp']}:{sc['cost_fn']}",
        'iForest Val Thresh': f"{sc['isolation_forest']['val_min_cost_threshold']:.4f}",
        'iForest Test Cost': f"{sc['isolation_forest']['test_cost']:.0f}",
        'LOF Val Thresh': f"{sc['local_outlier_factor']['val_min_cost_threshold']:.4f}",
        'LOF Test Cost': f"{sc['local_outlier_factor']['test_cost']:.0f}"
    })
display(pd.DataFrame(cost_rows))

## 39. Selected Canonical Operating Points (Validation Partition)

Rather than relying solely on maximum F1, four operational points were selected on the **Validation set**:
1. **High Precision / Low False Alarm (`low_false_alarm`):** Minimizes cardholder friction (target $\text{FPR} \le 0.15\%$).
2. **Balanced Point (`balanced`):** Maximizes the validation F1-harmonic mean.
3. **High Recall Point (`high_recall`):** Targets $\ge 70\%$ recall for aggressive threat interception.
4. **Controlled Low FPR Point (`low_fpr`):** Strictly enforces $\text{FPR} \le 0.25\%$.

## 40. Final Evaluation on Untouched Test Partition (N=56,962)

Each validation-selected threshold was evaluated **exactly once** on the untouched Test partition (N=56,962 transactions, 99 authentic fraud cases).
No thresholds were adjusted based on test results.

In [ ]:
# Display Test Operating Points comparison chart
display(Image(filename='outputs/figures/test_operating_points.png', width=800))

# Display complete test performance table
test_summary = []
for model_key, model_label in [('isolation_forest', 'Isolation Forest'), ('local_outlier_factor', 'Local Outlier Factor')]:
    eval_dict = operating_points[model_key]['untouched_test_evaluation']
    for mode, metrics in eval_dict.items():
        test_summary.append({
            'Model': model_label,
            'Operating Mode': metrics['name'],
            'Threshold': f"{metrics['threshold']:.4f}",
            'TP': metrics['test_tp'],
            'FP': metrics['test_fp'],
            'FN': metrics['test_fn'],
            'Precision': f"{metrics['test_precision']*100:.2f}%",
            'Recall': f"{metrics['test_recall']*100:.2f}%",
            'F1': f"{metrics['test_f1']:.4f}",
            'FPR': f"{metrics['test_fpr']*100:.3f}%"
        })
display(pd.DataFrame(test_summary))

## 41. Comprehensive Architectural & Empirical Comparison

Comparing Isolation Forest and Local Outlier Factor along all technical dimensions:

| Dimension | Isolation Forest | Local Outlier Factor | Operational Impact |
| :--- | :---: | :---: | :--- |
| **Test PR-AUC** | 0.1081 ($62\times$ base) | **0.2186** ($126\times$ base) | LOF provides $2.02\times$ higher overall ranking quality |
| **Balanced F1** | 0.1993 | **0.3455** | LOF provides $+73.4\%$ higher peak balanced F1 |
| **Balanced Recall** | 29.29% | **57.58%** | LOF captures $1.97\times$ more frauds at comparable FP levels |
| **Max Recall Mode** | 77.78% (1,474 FP) | **81.82%** (537 FP) | LOF achieves higher recall with $63.6\%$ fewer false alarms |
| **Low-FPR Mode** | 24.24% recall (121 FP) | **51.52%** recall (115 FP) | LOF detects more than double the fraud under FPR $\le 0.25\%$ |
| **Test Batch Latency** | **1.01 sec** | 11.36 sec | Isolation Forest is **$11.3\times$ faster** |
| **Per-Tx Latency** | **~0.018 ms** | ~0.200 ms | Both easily satisfy sub-10ms SLA, but iForest scales better |
| **Model Artifact Size** | **1.84 MB** | 185.80 MB | LOF requires storing full reference training manifold |

In [ ]:
# Display final model comparison summary figure
display(Image(filename='outputs/figures/final_model_comparison.png', width=800))

## 42. Final Model Recommendation & Business Trade-off Analysis

### Nuanced, Evidence-Based Recommendation
A single unconditional model recommendation is operationally sub-optimal. The empirical evidence supports a **Conditional Two-Tier Architecture**:

1. **Tier 1 — Real-Time Inline Gateway Interceptor: Isolation Forest**
   * **Operational Role:** Synchronous transaction authorization (scoring latency < 1 ms, minimal RAM footprint of 1.84 MB).
   * **Recommended Operating Point:** `Low False Alarm` ($\tau = 0.6722$, $\text{FPR} = 0.077\%$) or `Controlled Low FPR` ($\tau = 0.6341$, $\text{FPR} = 0.213\%$).
   * **Business Value:** Guarantees near-zero cardholder insult (only 44 to 121 false alerts across 56,962 transactions) while filtering obvious structural anomalies.

2. **Tier 2 — Asynchronous High-Sensitivity Review Queue: Local Outlier Factor**
   * **Operational Role:** Near-line fraud screening and compliance investigation (higher RAM footprint of 185.8 MB, 11s batch latency).
   * **Recommended Operating Point:** `Balanced` ($\tau = 2.4901$, Recall = 57.6%, 174 FPs) or `Controlled Low FPR` ($\tau = 2.7073$, Recall = 51.5%, 115 FPs).
   * **Business Value:** Outperforms Isolation Forest on all fraud-capture metrics (PR-AUC 0.2186 vs 0.1081), capturing 51–57 fraud cases with minimal false positive burden.

### Limitations & Pre-Production Boundary
* **Unsupervised Assumptions:** Neither model guarantees 100% fraud interception without supervised labels or calibrated transaction risk scoring.
* **Dataset Non-Stationarity:** Concept drift and adversarial behavioral shifts in production require periodic retraining and rolling quantile threshold re-calibration.
* **System boundary:** This system demonstrates algorithmic feasibility under rigorous scientific protocol, but is not certified for direct production deployment without operational load testing and human-in-the-loop review queues.